# Lecture 10: Link Prediction with Graph Neural Networks — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise message passing: the vanilla GNN layer and its $(A+I)HW$ matrix form, sum / mean / symmetric normalisation, permutation invariance, graph attention, over-smoothing and the concatenation / gated updates that fight it, and finally a small GNN link predictor trained from scratch in NumPy, including an **inductive** test on a graph it has never seen. The self-study used PyTorch Geometric; here every layer is a few lines of NumPy on adjacency matrices.

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

# Zachary's Karate Club (same 0-indexed edge list as networkx.karate_club_graph(), 34 nodes, 78 edges)
KARATE_EDGES = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 10), (0, 11), (0, 12), (0, 13),
    (0, 17), (0, 19), (0, 21), (0, 31), (1, 2), (1, 3), (1, 7), (1, 13), (1, 17), (1, 19), (1, 21), (1, 30), (2, 3),
    (2, 7), (2, 8), (2, 9), (2, 13), (2, 27), (2, 28), (2, 32), (3, 7), (3, 12), (3, 13), (4, 6), (4, 10), (5, 6),
    (5, 10), (5, 16), (6, 16), (8, 30), (8, 32), (8, 33), (9, 33), (13, 33), (14, 32), (14, 33), (15, 32), (15, 33),
    (18, 32), (18, 33), (19, 33), (20, 32), (20, 33), (22, 32), (22, 33), (23, 25), (23, 27), (23, 29), (23, 32),
    (23, 33), (24, 25), (24, 27), (24, 31), (25, 31), (26, 29), (26, 33), (27, 33), (28, 31), (28, 33), (29, 32),
    (29, 33), (30, 32), (30, 33), (31, 32), (31, 33), (32, 33)]
# Ground-truth split after the club broke up: 0 = "Mr. Hi", 1 = "Officer"
KARATE_CLUB = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

def edges_to_adj(edges, n=None, directed=False):
    """Edge list -> dense 0/1 NumPy adjacency matrix."""
    n = n if n is not None else max(max(e) for e in edges) + 1
    A = np.zeros((n, n))
    for u, v in edges:
        A[u, v] = 1
        if not directed:
            A[v, u] = 1
    return A

A_karate = edges_to_adj(KARATE_EDGES, 34)

from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(10)
sigmoid = lambda x: 1 / (1 + np.exp(-x))
relu = lambda x: np.maximum(x, 0)

# Attributed graphs from a stochastic block model: 4 communities of 30 nodes, noisy 8-d features per community
_MU = np.random.default_rng(0).choice([-1.0, 1.0], size=(4, 8))
def make_sbm(seed, n=120, p_in=0.2, p_out=0.02, noise=1.5):
    r = np.random.default_rng(seed)
    blk = np.repeat(np.arange(4), n // 4)
    P = np.where(blk[:, None] == blk[None, :], p_in, p_out)
    A = np.triu((r.random((n, n)) < P).astype(float), 1); A = A + A.T
    X = _MU[blk] + noise * r.normal(size=(n, 8))
    return A, X, blk

def split_edges(A, frac=0.2, seed=0):
    """Hide a fraction of edges. Returns (A_train, train_edges, hidden_edges, non_edges) as int arrays of pairs."""
    r = np.random.default_rng(seed)
    iu, ju = np.triu_indices(len(A), 1)
    E = np.c_[iu, ju][A[iu, ju] > 0]
    idx = r.permutation(len(E)); nh = int(frac * len(E))
    hidden, train = E[idx[:nh]], E[idx[nh:]]
    A_tr = np.zeros_like(A); A_tr[train[:, 0], train[:, 1]] = 1; A_tr = A_tr + A_tr.T
    return A_tr, train, hidden, np.c_[iu, ju][A[iu, ju] == 0]

A_sbm, X_sbm, blk_sbm = make_sbm(seed=1)
A_sbm_tr, E_train, E_hidden, E_non = split_edges(A_sbm)

## Part A · Concepts

### Exercise 1 · What GNNs fix about shallow embeddings
*🧠 Concept · ★☆☆*

Lecture 9 ended with three limitations of shallow embeddings. For each, explain which part of the GNN encoder
$\text{ENC}(u) = h_u^{(K)}$ fixes it. Then explain why initialising $h_u^{(0)}$ with **one-hot node identities** brings one of the problems back.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Where do the parameters of a GNN live: per node, or per layer? What does $h_u^{(0)} = x_u$ allow?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **No parameter sharing** → the weights $W^{(k)}$ are shared by all nodes, so the parameter count depends on feature dims and depth, not on $|V|$.
2. **No features** → $h_u^{(0)} = x_u$ feeds node attributes (e.g. CORA's 1433-d bag of words) straight into the encoder.
3. **Transductive** → a new node only needs its features and neighbours. Running the same layers produces its embedding with no retraining:
   the model is **inductive**.

With one-hot identities, $x_u\in\{0,1\}^{|V|}$, the first weight matrix $W^{(1)}\in\mathbb R^{d\times|V|}$ has one column per node, which is
again a lookup table. A new node has no column, so the model is transductive again (Critical Insight 4.1).

</details>

### Exercise 2 · Which aggregators are permutation invariant?
*🧠 Concept · ★☆☆*

$\mathcal N(u)$ is an unordered set. Which of these AGGREGATE functions are permutation invariant?

(a) $\sum_v h_v$ (b) $\max_v h_v$ (element-wise) (c) $\text{MLP}([h_{v_1}\parallel h_{v_2}\parallel\dots])$ in storage order
(d) $\text{MLP}_\theta(\sum_v\text{MLP}_\phi(h_v))$ (e) an LSTM over neighbours in a random order (f) Janossy pooling averaging $\rho_\phi$ over **all** permutations.

Why is invariance required, and what's the practical compromise of Janossy pooling?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: does relabelling the neighbours change the output?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Invariant: (a), (b), (d) (Deep Sets: a sum of per-element transforms followed by an MLP), (f). Not invariant: (c) depends on the storage
order and also needs a fixed number of neighbours; (e) depends on the sampled order (it's only invariant *in expectation*).

Node ids are arbitrary: the same graph stored with a different node ordering must give the same embeddings (up to the same relabelling),
otherwise the model learns artefacts of the file format. Janossy pooling makes any sequence model $\rho_\phi$ invariant by averaging over
permutations, but there are $|\mathcal N(u)|!$ of them. In practice one samples a few ($\Pi$), which makes the output only approximately invariant.

</details>

### Exercise 3 · Reading the self-study code critically
*🧠 Concept · ★★☆*

Compare the self-study's PyG code with the theory:

1. In `GCNConv.forward` the order is `lin` → `ReLU` → `propagate` (sum over $A+I$). Write this as a matrix formula and compare with
   $H^{(t)} = \sigma((A+I)H^{(t-1)}W^{(t)})$. Are they the same?
2. `MyGnn` ends with `Linear` → `Sigmoid` and then `h @ h.T`. What range can the scores take? Is MSE against a 0/1 adjacency a sensible loss?
3. The note blames the poor CORA result (HitRate@10 ≈ 0.5%) on `aggr='add'`. Explain the mechanism.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Is $\text{ReLU}$ linear? Can $\text{ReLU}((A+I)HW)$ differ from $(A+I)\text{ReLU}(HW)$? What is $h_u^\top h_v$ when all entries are in $(0,1)$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Code: $H^{(t)} = (A+I)\,\text{ReLU}(H^{(t-1)}W)$; theory: $\text{ReLU}((A+I)H^{(t-1)}W)$. They differ because ReLU is non-linear:
   e.g. two neighbours with transformed values $+1$ and $-1$ give $\text{ReLU}(0) = 0$ in theory but $1 + 0 = 1$ in the code. The code's version
   is a valid message-passing layer (the non-linearity sits in the *message*), just not the formula on the slide, and its outputs are
   unnormalised non-negative sums.
2. Sigmoid outputs lie in $(0,1)^d$, so $h_u^\top h_v\in(0,d)$: every pair gets a positive score, and only the magnitudes differ. MSE against 0/1
   targets with ~95% zeros mostly teaches "predict small numbers". A logistic (BCE) loss on $\sigma(h_u^\top h_v)$ with negative sampling is the
   standard choice; that's what Exercise 16 uses.
3. With sum aggregation, $\lVert h_u\rVert$ grows with the degree and multiplies across layers: hub embeddings are orders of magnitude larger
   than leaf embeddings, so dot products are dominated by hubs and gradients explode. Mean or symmetric normalisation
   (Exercise 7) keeps the scales comparable.

</details>

### Exercise 4 · Receptive fields and over-smoothing
*🧠 Concept · ★★☆*

1. The self-study improved HitRate@10 on karate from $25\%$ (`[2,3,2]`, 1 conv layer) to $41.7\%$ (`[2,8,4,2]`, 2 layers). Explain using the
   notion of a receptive field / computation tree.
2. Why doesn't "just stack 20 layers" keep improving things? Describe over-smoothing.
3. How do the **concatenation** and **gated interpolation** updates counteract it?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

After $K$ layers $h_u$ depends on the $K$-hop neighbourhood. Karate has diameter 5. Repeated averaging converges to what?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $K$ message-passing layers unroll into a depth-$K$ computation tree, so $h_u^{(K)}$ sees the $K$-hop neighbourhood. For link prediction
   two nodes at distance 2 share neighbours that a 1-layer model can't "see" from both ends; with 2 layers $h_u$ and $h_v$ both contain information
   about their common neighbours (cf. common-neighbour heuristics in Lecture 8).
2. Each layer is (roughly) a smoothing/averaging operator. Repeating it $K$ times is like a $K$-step random walk: as $K\to\infty$ every node's
   representation converges to the same stationary average (Exercise 10), so embeddings become indistinguishable and the decoder scores
   everything equally. On karate (diameter 5) a handful of layers already cover the whole graph.
3. **Concatenation** $[\text{UPDATE}_{base}(h_u, m)\parallel h_u]$ keeps the node's previous state as separate coordinates that never get averaged.
   **Gating** $\alpha_1\circ\text{UPDATE}_{base} + (1-\alpha_1)\circ h_u$ lets the model learn per dimension how much neighbourhood information to mix in,
   like a residual connection (or a GRU-style update), so the node's own signal decays slowly instead of being washed out.

</details>

### Exercise 5 · Content vs structure
*🧠 Concept · ★☆☆*

You do link prediction on (i) CORA with 1433-d bag-of-words features and (ii) a social network with **no** node attributes.
What would you use as $h_u^{(0)}$ in each case, and which of your choices keeps the model inductive? When does normalising the aggregation
matter most (note box in section 2.3.1)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Structural features: degree, clustering coefficient, PageRank, Laplacian eigenvectors, a constant vector…

</details>

<details>
<summary><b>✅ Solution</b></summary>

(i) The bag-of-words vectors: content-driven, inductive (a new paper has a text).
(ii) Structural features computed from the graph: degree, log-degree, local clustering, PageRank, or even a constant vector $\mathbf 1$
(then the GNN learns purely from structure). These are inductive because they can be computed for a new node. One-hot ids are not.
Normalisation matters most when the node's own features carry more signal than the structure, and when degrees vary widely (scale-free graphs),
since otherwise hubs dominate the sums.

</details>

## Part B · By hand

### Exercise 6 · One vanilla GNN layer
*✍️ By hand · ★☆☆*

Path graph $0-1-2$ with scalar features $h^{(0)} = (1, 2, 3)$. Apply the basic layer
$h_u^{(1)} = \text{ReLU}\big(W_{self}h_u + W_{neigh}\sum_{v\in\mathcal N(u)}h_v + b\big)$ with $W_{self} = 2$, $W_{neigh} = 1$, $b = -4$.
Then compute the self-loop form $(A+I)h^{(0)}$ (no weights, no activation).

In [ ]:
h1_vanilla = None   # [h_0, h_1, h_2]
h1_selfloop = None

_A = edges_to_adj([(0, 1), (1, 2)], 3); _h = np.array([1., 2, 3])
check('vanilla layer', h1_vanilla, relu(2 * _h + _A @ _h - 4))
check('(A+I)h', h1_selfloop, (_A + np.eye(3)) @ _h)

<details>
<summary><b>💡 Hint</b></summary>

Neighbour sums: node 0 → 2, node 1 → 1 + 3 = 4, node 2 → 2.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Node 0: $2\cdot1 + 2 - 4 = 0$; node 1: $2\cdot2 + 4 - 4 = 4$; node 2: $2\cdot 3 + 2 - 4 = 4$ → $h^{(1)} = (0, 4, 4)$.
$(A+I)h = (1+2,\ 1+2+3,\ 2+3) = (3, 6, 5)$: the self-loop form is the special case $W_{self} = W_{neigh}$.

```python
h1_vanilla = [0, 4, 4]
h1_selfloop = [3, 6, 5]
```

</details>

### Exercise 7 · Sum, mean and symmetric normalisation
*✍️ By hand · ★★☆*

Star graph: centre 0 connected to leaves 1, 2, 3 (no self-loops). Features $h = (4, 1, 2, 3)$. Compute the message $m_{\mathcal N(u)}$ for the
centre $u=0$ and for leaf $u=1$ under (a) sum, (b) mean, (c) symmetric normalisation $\sum_v h_v/\sqrt{|\mathcal N(u)||\mathcal N(v)|}$.

In [ ]:
m_centre = None   # [sum, mean, sym]
m_leaf = None     # [sum, mean, sym]

_A = edges_to_adj([(0, 1), (0, 2), (0, 3)], 4); _h = np.array([4., 1, 2, 3]); _d = _A.sum(1)
_S = _A / np.sqrt(np.outer(_d, _d))
check('centre messages', m_centre, [(_A @ _h)[0], (_A @ _h / _d)[0], (_S @ _h)[0]], tol=1e-3)
check('leaf messages', m_leaf, [(_A @ _h)[1], (_A @ _h / _d)[1], (_S @ _h)[1]], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$|\mathcal N(0)| = 3$, $|\mathcal N(\text{leaf})| = 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Centre: sum $1+2+3 = 6$; mean $2$; symmetric $6/\sqrt{3\cdot1} = 3.464$.
Leaf 1: sum $4$; mean $4$; symmetric $4/\sqrt{1\cdot3} = 2.309$.
Sum lets the hub's message scale with its degree; mean removes that completely; symmetric normalisation is in between, and it also down-weights
messages *from* hubs (the leaf hears the centre at $1/\sqrt3$ strength). In matrix form: $D^{-1/2}AD^{-1/2}H$.

```python
m_centre = [6, 2, 6 / np.sqrt(3)]
m_leaf = [4, 4, 4 / np.sqrt(3)]
```

</details>

### Exercise 8 · Attention weights by hand
*✍️ By hand · ★★☆*

Node $u$ has neighbours $v_1, v_2, v_3$. After the shared transform the (scalar) values are $Wh_u = 1$, $Wh_{v_1} = 1$, $Wh_{v_2} = 2$, $Wh_{v_3} = -1$.
The attention vector is $\mathbf a = (1, 1)$, LeakyReLU slope $0.2$. Compute $\alpha_{u,v}$ for all three neighbours and the message
$m = \sum_v \alpha_{u,v}\,Wh_v$.

In [ ]:
alpha_att = None   # [a1, a2, a3]
m_att = None

_e = np.array([1 + 1, 1 + 2, 1 - 1.]); _e = np.where(_e > 0, _e, 0.2 * _e)
_al = np.exp(_e) / np.exp(_e).sum()
check('attention weights', alpha_att, _al, tol=1e-4); check('message', m_att, _al @ np.array([1, 2, -1.]), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$e_v = \text{LeakyReLU}(\mathbf a^\top[Wh_u\parallel Wh_v]) = \text{LeakyReLU}(1 + Wh_v)$: scores $2, 3, 0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Scores $e = (2, 3, 0)$ (all $\ge 0$, so LeakyReLU does nothing). Softmax: $e^2 = 7.389$, $e^3 = 20.086$, $e^0 = 1$, sum $28.475$ →
$\alpha = (0.2595, 0.7054, 0.0351)$. Message $m = 0.2595 + 2(0.7054) - 0.0351 = 1.6352$.
Note that with $\mathbf a = (1,1)$ the $u$-part of the score is the same for all neighbours and cancels in the softmax; the attention here only
depends on $Wh_v$. Real GATs learn $\mathbf a$ and use several heads.

```python
_e = np.array([2., 3., 0.])
alpha_att = np.exp(_e) / np.exp(_e).sum()
m_att = alpha_att @ np.array([1., 2., -1.])
```

</details>

### Exercise 9 · Counting parameters and computation trees
*✍️ By hand · ★☆☆*

1. The self-study's CORA model `[1433, 256, 64, 32]` uses bias-free GCN layers. How many weights does it have? Compare with a shallow
   embedding table for CORA's 2708 nodes with $d = 32$, and say which number grows when CORA doubles in size.
2. In a 3-regular graph, how many nodes (counted with repetition) does the unrolled computation tree of a 2-layer GNN contain for one target
   node (no self-loops)?

In [ ]:
gnn_params = None
shallow_params = None
tree_size = None

check('GNN weights', gnn_params, 1433 * 256 + 256 * 64 + 64 * 32)
check('shallow parameters', shallow_params, 2708 * 32)
check('computation tree size', tree_size, 1 + 3 + 3 * 3)

<details>
<summary><b>💡 Hint</b></summary>

A layer $d_{in}\to d_{out}$ has $d_{in}d_{out}$ weights. The tree: root, its 3 neighbours, each with its own 3 neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $1433\cdot256 + 256\cdot64 + 64\cdot32 = 366\,848 + 16\,384 + 2\,048 = 385\,280$ weights vs $2708\cdot32 = 86\,656$ for the lookup table.
   The GNN is bigger here (because of the 1433-d input), but its size is **independent of $|V|$**: doubling CORA doubles the shallow table and
   leaves the GNN unchanged.
2. $1 + 3 + 3\cdot3 = 13$ (the tree includes paths that go back to the root, since messages flow both ways). In general the tree has
   $O(d^K)$ nodes, which is why deep GNNs on dense graphs need neighbour sampling (GraphSAGE).

```python
gnn_params = 1433 * 256 + 256 * 64 + 64 * 32
shallow_params = 2708 * 32
tree_size = 13
```

</details>

### Exercise 10 · Where does over-smoothing converge?
*✍️ By hand · ★★★*

Mean aggregation with self-loops is $H \leftarrow \tilde D^{-1}\tilde A H$ with $\tilde A = A + I$. On a connected graph
$\tilde P = \tilde D^{-1}\tilde A$ is the transition matrix of an ergodic random walk (Lecture 8).

1. Show that $\tilde P^k\mathbf x \to (\pi^\top\mathbf x)\mathbf 1$ where $\pi$ is the stationary distribution, and that $\pi_i\propto\tilde d_i$.
2. For the path $0-1-2$ and $\mathbf x = (1, 2, 3)$, compute the limit value.

In [ ]:
oversmooth_limit = None   # the common value all nodes converge to

_At = edges_to_adj([(0, 1), (1, 2)], 3) + np.eye(3)
_P = _At / _At.sum(1, keepdims=True)
check('limit', oversmooth_limit, (np.linalg.matrix_power(_P, 500) @ np.array([1., 2, 3]))[0], tol=1e-6)

<details>
<summary><b>💡 Hint 1</b></summary>

Ergodic chain: $\tilde P^k \to \mathbf 1\pi^\top$ (every row becomes $\pi^\top$).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Detailed balance: $\tilde d_i\tilde P_{ij} = \tilde A_{ij} = \tilde d_j\tilde P_{ji}$, so $\pi\propto\tilde d$. Here $\tilde d = (2, 3, 2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. For an ergodic chain every row of $\tilde P^k$ converges to $\pi^\top$, so $\tilde P^k\mathbf x\to\mathbf 1\,\pi^\top\mathbf x$: **every** node ends
   up with the same value, which is over-smoothing. Since $\tilde A$ is symmetric, $\tilde d_i\tilde P_{ij} = \tilde d_j\tilde P_{ji}$ (detailed balance),
   so $\pi_i = \tilde d_i/\sum_j\tilde d_j$.
2. $\pi = (2,3,2)/7$, limit $= (2\cdot1 + 3\cdot2 + 2\cdot3)/7 = 14/7 = 2$.
The convergence speed is set by the second-largest eigenvalue modulus of $\tilde P$: the better connected the graph, the faster it smooths.

```python
oversmooth_limit = 2.0
```

</details>

### Exercise 11 · Gated interpolation update
*✍️ By hand · ★☆☆*

$\text{UPDATE}_{base}(h_u, m) = (2, -1)$, $h_u = (1, 1)$, and the gate is $\alpha_1 = (0.3, 0.9)$, $\alpha_2 = 1-\alpha_1$. Compute the
interpolated update. Also give the **concatenation** update $[\text{UPDATE}_{base}\parallel h_u]$. Which dimension of the gated output is
mostly "self" and which mostly "neighbourhood"?

In [ ]:
h_gated = None
h_concat = None

_a1 = np.array([.3, .9])
check('gated update', h_gated, _a1 * np.array([2, -1.]) + (1 - _a1) * np.array([1, 1.]))
check('concat update', h_concat, np.r_[[2, -1.], [1, 1.]])

<details>
<summary><b>💡 Hint</b></summary>

Element-wise: $\alpha_1\circ(2,-1) + \alpha_2\circ(1,1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(0.3\cdot2 + 0.7\cdot1,\ 0.9\cdot(-1) + 0.1\cdot1) = (1.3, -0.8)$. Concatenation: $(2, -1, 1, 1)$, which doubles the dimension.
Dimension 1 is 70% self (gate 0.3 on the neighbourhood update); dimension 2 is 90% neighbourhood.

```python
h_gated = [1.3, -0.8]
h_concat = [2, -1, 1, 1]
```

</details>

## Part C · Code from scratch

### Exercise 12 · A message-passing layer in matrix form
*💻 Code · ★☆☆*

Implement `gnn_layer(A, H, W, norm='sum', self_loops=True, act=relu)` computing $\sigma(\hat A H W)$ with

* `'sum'`: $\hat A = \tilde A$; `'mean'`: $\hat A = \tilde D^{-1}\tilde A$; `'sym'`: $\hat A = \tilde D^{-1/2}\tilde A\tilde D^{-1/2}$,

where $\tilde A = A + I$ if `self_loops` else $A$. Run it on karate with random features and store the three outputs.

In [ ]:
def gnn_layer(A, H, W, norm='sum', self_loops=True, act=relu):
    # TODO
    return None

H0 = rng.normal(size=(34, 5)); W0 = rng.normal(size=(5, 3))
out_sum = gnn_layer(A_karate, H0, W0, 'sum')
out_mean = gnn_layer(A_karate, H0, W0, 'mean')
out_sym = gnn_layer(A_karate, H0, W0, 'sym')

# reference: per-node loops straight from the message-passing definitions
_N = [list(np.flatnonzero(A_karate[u])) + [u] for u in range(34)]
_ref_sum = np.array([relu(sum(H0[v] for v in _N[u]) @ W0) for u in range(34)])
_ref_mean = np.array([relu(sum(H0[v] for v in _N[u]) / len(_N[u]) @ W0) for u in range(34)])
_ref_sym = np.array([relu(sum(H0[v] / np.sqrt(len(_N[u]) * len(_N[v])) for v in _N[u]) @ W0) for u in range(34)])
check('sum aggregation', out_sum, _ref_sum); check('mean aggregation', out_mean, _ref_mean); check('symmetric', out_sym, _ref_sym)

<details>
<summary><b>💡 Hint</b></summary>

`At = A + np.eye(n)`; `d = At.sum(1)`; `'mean'`: `At / d[:, None]`; `'sym'`: `At / np.sqrt(np.outer(d, d))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

One matrix product replaces the loop over nodes; with sparse $\hat A$ the cost is $O(m\,d)$ per layer. Symmetric normalisation with self-loops
is exactly Kipf & Welling's GCN propagation rule. Compare the row norms of `out_sum` for node 33 (degree 17) and a leaf: the sum version
lets hubs dominate.

```python
def gnn_layer(A, H, W, norm='sum', self_loops=True, act=relu):
    At = A + np.eye(len(A)) if self_loops else A.copy()
    d = At.sum(1)
    if norm == 'mean':
        At = At / d[:, None]
    elif norm == 'sym':
        At = At / np.sqrt(np.outer(d, d))
    return act(At @ H @ W)

H0 = rng.normal(size=(34, 5)); W0 = rng.normal(size=(5, 3))
out_sum = gnn_layer(A_karate, H0, W0, 'sum')
out_mean = gnn_layer(A_karate, H0, W0, 'mean')
out_sym = gnn_layer(A_karate, H0, W0, 'sym')
print('row norm hub vs leaf (sum):', np.linalg.norm(out_sum[33]).round(2), np.linalg.norm(out_sum[11]).round(2))
```

</details>

### Exercise 13 · Permutation equivariance test
*💻 Code · ★☆☆*

A GNN layer must be **equivariant**: relabelling the nodes with a permutation matrix $\Pi$ permutes the output the same way,
$f(\Pi A\Pi^\top, \Pi H) = \Pi f(A, H)$. Test this numerically for your `gnn_layer` (all three norms) with a random permutation, and store
`True`/`False` in `equivariant`. Then show that a "layer" that flattens $H$ and applies a dense matrix (an MLP on the whole graph) is **not**
equivariant: store the result in `mlp_equivariant`.

In [ ]:
perm = rng.permutation(34)
Pi = np.eye(34)[perm]
equivariant = None
mlp_equivariant = None

check('GNN layer is equivariant', equivariant, True)
check('flat MLP is not', mlp_equivariant, False)

<details>
<summary><b>💡 Hint</b></summary>

Compare `gnn_layer(Pi @ A @ Pi.T, Pi @ H0, W0, norm)` with `Pi @ gnn_layer(A, H0, W0, norm)` using `np.allclose`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Every operation in the GNN layer is either node-wise ($HW$, the activation) or a product with the adjacency, and
$(\Pi A\Pi^\top)(\Pi H) = \Pi(AH)$ because $\Pi^\top\Pi = I$. The flat MLP mixes coordinate $i$ of the input with fixed weights tied to
*position* $i$, so it depends on the arbitrary node order. Graph-level read-outs (sum/mean over nodes, Lecture 11) are then permutation **invariant**.

```python
perm = rng.permutation(34)
Pi = np.eye(34)[perm]
equivariant = all(np.allclose(gnn_layer(Pi @ A_karate @ Pi.T, Pi @ H0, W0, nm), Pi @ gnn_layer(A_karate, H0, W0, nm))
                  for nm in ('sum', 'mean', 'sym'))
W_flat = rng.normal(size=(34 * 5, 34 * 3))
mlp = lambda H: relu(H.reshape(-1) @ W_flat).reshape(34, 3)
mlp_equivariant = bool(np.allclose(mlp(Pi @ H0), Pi @ mlp(H0)))
print(equivariant, mlp_equivariant)
```

</details>

### Exercise 14 · Watching over-smoothing happen
*💻 Code · ★★☆*

On karate, start from random features $H^{(0)}\in\mathbb R^{34\times4}$ and apply $K$ weight-free mean-aggregation layers
$H\leftarrow\tilde D^{-1}\tilde AH$. Measure the **spread** $\text{sp}(H) = \frac1n\sum_u\lVert h_u - \bar h\rVert^2$ for $K = 0,1,2,4,8,16,32$
(`spread_plain`). Repeat with a **gated / residual** update $H\leftarrow 0.5\,\tilde D^{-1}\tilde AH + 0.5\,H^{(0)}$ (skip connection to the input,
`spread_gated`). What happens in each case?

In [ ]:
Ks = [0, 1, 2, 4, 8, 16, 32]
H_init = np.random.default_rng(0).normal(size=(34, 4))
spread_plain = None   # list, one value per K
spread_gated = None

if spread_plain is not None:
    check('plain spread decreases monotonically', bool(np.all(np.diff(spread_plain) < 0)), True)
    check('plain spread collapses (K=32 < 1e-3 * K=0)', spread_plain[-1] < 1e-3 * spread_plain[0], True)
    check('gated update keeps > 10% of spread', spread_gated[-1] > 0.1 * spread_gated[0], True)
else:
    check('over-smoothing', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`P = (A + I) / (A + I).sum(1, keepdims=True)`; spread: `np.mean(np.sum((H - H.mean(0))**2, 1))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Plain mean aggregation shrinks the spread geometrically: after ~30 layers all 34 embeddings are essentially identical (the stationary average
of Exercise 10). With the skip connection the iteration converges instead to $H^\star = 0.5(I - 0.5\tilde P)^{-1}H^{(0)}$, a smoothed
version that keeps node-specific information. This is the personalised-PageRank idea behind APPNP, and the intuition behind gated and
concatenation updates.

```python
Ks = [0, 1, 2, 4, 8, 16, 32]
H_init = np.random.default_rng(0).normal(size=(34, 4))
At = A_karate + np.eye(34); P = At / At.sum(1, keepdims=True)
sp = lambda H: np.mean(np.sum((H - H.mean(0))**2, 1))
spread_plain, spread_gated = [], []
for K in Ks:
    H = H_init.copy(); G = H_init.copy()
    for _ in range(K):
        H = P @ H
        G = 0.5 * P @ G + 0.5 * H_init
    spread_plain.append(sp(H)); spread_gated.append(sp(G))
print(np.round(spread_plain, 4)); print(np.round(spread_gated, 4))
```

</details>

### Exercise 15 · A graph attention layer
*💻 Code · ★★☆*

Implement a single-head GAT layer `gat_layer(A, H, W, a, slope=0.2)` returning `(H_new, alpha)` with
$\alpha_{u,v} = \text{softmax}_{v\in\mathcal N(u)\cup\{u\}}\big(\text{LeakyReLU}(\mathbf a^\top[Wh_u\parallel Wh_v])\big)$ and
$h'_u = \sum_v\alpha_{u,v}Wh_v$ (no activation). Use self-loops. Vectorise: with $\mathbf a = [\mathbf a_1\parallel\mathbf a_2]$ the score
matrix is $e_{uv} = (\mathbf a_1^\top Wh_u) + (\mathbf a_2^\top Wh_v)$. Mask non-edges with $-\infty$ before the softmax.
Note: here $W$ acts on rows, so compute $HW$ with $W\in\mathbb R^{d\times d'}$.

In [ ]:
def gat_layer(A, H, W, a, slope=0.2):
    # TODO: return H_new, alpha (n x n, rows sum to 1, zero off the neighbourhood)
    return None, None

W_g = rng.normal(size=(5, 3)); a_g = rng.normal(size=6)
H_gat, alpha_gat = gat_layer(A_karate, H0, W_g, a_g)

if alpha_gat is not None:
    _Z = H0 @ W_g
    _ref = np.zeros((34, 3)); _alr = np.zeros((34, 34))
    for u in range(34):
        nb_ = list(np.flatnonzero(A_karate[u])) + [u]
        e = np.array([a_g @ np.r_[_Z[u], _Z[v]] for v in nb_]); e = np.where(e > 0, e, 0.2 * e)
        w = np.exp(e - e.max()); w /= w.sum()
        _alr[u, nb_] = w; _ref[u] = w @ _Z[nb_]
    check('attention matrix', alpha_gat, _alr); check('GAT output', H_gat, _ref)
else:
    check('GAT layer', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

`Z = H @ W`; `e = (Z @ a[:d'])[:, None] + (Z @ a[d':])[None, :]`; LeakyReLU: `np.where(e > 0, e, slope * e)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`e = np.where(At > 0, e, -np.inf)`; subtract the row max before `exp` for stability.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Attention lets each node weight neighbours by learned relevance instead of degree. The mean aggregator is the special case $\mathbf a = 0$
(uniform weights). The additive score form makes GAT's scoring "static": the *ranking* of neighbours by $\mathbf a_2^\top Wh_v$ is the same for every $u$,
which GATv2 fixes by applying the non-linearity before $\mathbf a$.

```python
def gat_layer(A, H, W, a, slope=0.2):
    At = A + np.eye(len(A))
    Z = H @ W; dp = Z.shape[1]
    e = (Z @ a[:dp])[:, None] + (Z @ a[dp:])[None, :]
    e = np.where(e > 0, e, slope * e)
    e = np.where(At > 0, e, -np.inf)
    w = np.exp(e - e.max(1, keepdims=True))
    alpha = w / w.sum(1, keepdims=True)
    return alpha @ Z, alpha

W_g = rng.normal(size=(5, 3)); a_g = rng.normal(size=6)
H_gat, alpha_gat = gat_layer(A_karate, H0, W_g, a_g)
```

</details>

### Exercise 16 · Train a GNN link predictor from scratch
*💻 Code · ★★★*

On the attributed SBM graph (`A_sbm`, features `X_sbm`, training graph `A_sbm_tr`, `E_train`, `E_hidden`, `E_non` from the setup) train a
**simplified GCN** (SGC) encoder $Z = \hat A^K X W$ with $\hat A = \tilde D^{-1/2}\tilde A\tilde D^{-1/2}$ of the *training* graph, and a dot-product
decoder $\sigma(\mathbf z_u^\top\mathbf z_v)$:

* each epoch: positives = `E_train`, negatives = as many uniformly random node pairs;
* loss = mean binary cross-entropy; gradient
  $\nabla_W = \frac1{|B|}\sum_{(u,v)}(\sigma(s_{uv}) - y_{uv})\,(\tilde x_u\tilde x_v^\top + \tilde x_v\tilde x_u^\top)W$ with $\tilde x = (\hat A^KX)_u$;
* full-batch gradient descent, `lr = 0.5`, 400 epochs, $W\in\mathbb R^{8\times4}$ initialised $\mathcal N(0, 0.1^2)$.

Write `train_sgc(A_tr, X, K, seed=0)` returning `W`, and `auc_link(Z)` = AUC of hidden edges vs. all non-edges. Store the AUC for $K=0$
(features only, no message passing) and $K=2$ in `auc_k0`, `auc_k2`.

In [ ]:
def propagate(A_tr, X, K):
    # TODO: return (D^-1/2 (A+I) D^-1/2)^K X
    return None

def train_sgc(A_tr, X, K, seed=0, lr=0.5, epochs=400, dim=4):
    # TODO
    return None

def auc_link(Z, hidden=E_hidden, non=E_non):
    # TODO
    return None

auc_k0 = None
auc_k2 = None

if auc_k2 is not None:
    check('message passing helps (AUC K=2 > K=0 + 0.05)', auc_k2 > auc_k0 + 0.05, True)
    check('K=2 AUC > 0.72', auc_k2 > 0.72, True)
    _Z = propagate(A_sbm_tr, X_sbm, 2) @ train_sgc(A_sbm_tr, X_sbm, 2)
    _s = lambda P: np.sum(_Z[P[:, 0]] * _Z[P[:, 1]], 1)
    check('auc_link matches sklearn', auc_link(_Z), roc_auc_score(np.r_[np.ones(len(E_hidden)), np.zeros(len(E_non))], np.r_[_s(E_hidden), _s(E_non)]))
else:
    check('SGC link prediction', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Vectorised gradient: with `Xu, Xv = Xt[pairs[:, 0]], Xt[pairs[:, 1]]` and `g = (sigmoid(s) - y) / len(y)`: `G = (Xu * g[:, None]).T @ (Xv @ W) + (Xv * g[:, None]).T @ (Xu @ W)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Negatives: `r.integers(0, n, size=(len(E_train), 2))`. A few might be real edges, which is fine (that's standard negative sampling noise).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Without message passing ($K=0$) the model can only use the noisy features: AUC ≈ 0.72. Two rounds of symmetric-normalised propagation
average each node's features with its (mostly same-community) neighbours, which denoises them, and the AUC rises to ≈ 0.79. Going to large $K$
eventually hurts (over-smoothing, Exercise 14). SGC drops the non-linearities between layers, so the whole encoder is linear in $W$ and the
gradient is easy to write by hand, but it is still a shared-weight, inductive GNN.

```python
def propagate(A_tr, X, K):
    At = A_tr + np.eye(len(A_tr)); d = At.sum(1)
    S = At / np.sqrt(np.outer(d, d))
    for _ in range(K):
        X = S @ X
    return X

def train_sgc(A_tr, X, K, seed=0, lr=0.5, epochs=400, dim=4, pos=None):
    pos = E_train if pos is None else pos
    r = np.random.default_rng(seed)
    Xt = propagate(A_tr, X, K)
    W = 0.1 * r.normal(size=(X.shape[1], dim))
    n = len(X)
    for _ in range(epochs):
        neg = r.integers(0, n, size=(len(pos), 2))
        pairs = np.vstack([pos, neg]); y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
        Xu, Xv = Xt[pairs[:, 0]], Xt[pairs[:, 1]]
        s = np.sum((Xu @ W) * (Xv @ W), 1)
        g = (sigmoid(s) - y) / len(y)
        W -= lr * ((Xu * g[:, None]).T @ (Xv @ W) + (Xv * g[:, None]).T @ (Xu @ W))
    return W

def auc_link(Z, hidden=E_hidden, non=E_non):
    pos = np.sum(Z[hidden[:, 0]] * Z[hidden[:, 1]], 1)
    neg = np.sum(Z[non[:, 0]] * Z[non[:, 1]], 1)
    return np.mean((pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :]))

W_k0 = train_sgc(A_sbm_tr, X_sbm, 0); W_k2 = train_sgc(A_sbm_tr, X_sbm, 2)
auc_k0 = auc_link(propagate(A_sbm_tr, X_sbm, 0) @ W_k0)
auc_k2 = auc_link(propagate(A_sbm_tr, X_sbm, 2) @ W_k2)
print('AUC K=0', auc_k0, ' K=2', auc_k2)
```

</details>

### Exercise 17 · Inductive: predict links on an unseen graph
*💻 Code · ★★★*

Shallow embeddings can't do this. Generate a **new** graph `A_new, X_new, _ = make_sbm(seed=2)` (same kind of communities and features,
different nodes and edges), split it with `split_edges(A_new, seed=1)`, and **without retraining** apply the $W$ you learnt on the first graph
($K=2$) to the new training graph's propagated features. Store the AUC on the new graph's hidden edges in `auc_inductive`.
How does it compare to `auc_k2`?

In [ ]:
A_new, X_new, _ = make_sbm(seed=2)
A_new_tr, _, E_new_hidden, E_new_non = split_edges(A_new, seed=1)
auc_inductive = None

check('inductive AUC > 0.72 on an unseen graph', None if auc_inductive is None else auc_inductive > 0.72, True)

<details>
<summary><b>💡 Hint</b></summary>

`Z_new = propagate(A_new_tr, X_new, 2) @ W_k2`; then `auc_link(Z_new, E_new_hidden, E_new_non)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The AUC on the never-seen graph (≈ 0.75) is only slightly below the training graph's (≈ 0.79) and far above chance (0.5): the learnt $W$ is a function of *features and neighbourhoods*,
not of node ids, so it transfers to new nodes and even new graphs drawn from the same distribution. That's the inductive property in the
comparison table of Critical Insight 4.3. A DeepWalk/SVD embedding table from Lecture 9 has no rows for these nodes at all.

```python
A_new, X_new, _ = make_sbm(seed=2)
A_new_tr, _, E_new_hidden, E_new_non = split_edges(A_new, seed=1)
W_k2 = train_sgc(A_sbm_tr, X_sbm, 2)
auc_inductive = auc_link(propagate(A_new_tr, X_new, 2) @ W_k2, E_new_hidden, E_new_non)
print('inductive AUC', auc_inductive)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*